# Quantum Oracle Sketching (QOS) — Quick Start

This notebook demonstrates the core QOS operations: state sketching, oracle sketching, and QSVT transforms.


In [1]:
import jax
import jax.numpy as jnp
from jax import random

# QOS imports
from qos.core.state_sketch import q_state_sketch_flat, q_state_sketch
from qos.core.oracle_sketch import (
    q_oracle_sketch_boolean,
    q_oracle_sketch_matrix_element,
    q_oracle_sketch_matrix_row_index,
)
from qos.primitives.amplification import amplitude_amplification
from qos.utils.numerical import random_flat_vector, random_unit_vector, random_sparse_matrix

key = random.PRNGKey(0)


## 1. Flat Vector State Sketching

For vectors with entries ±1, QOS constructs the state using a single ancilla qubit.

In [2]:
N = 1024
key, subkey = random.split(key)
flat_vector = random_flat_vector(subkey, N)

state, num_samples = q_state_sketch_flat(flat_vector, unit_num_samples=1_000_000)
error = jnp.linalg.norm(state - flat_vector / jnp.sqrt(N))
print(f"Norm error: {float(error):.3e} | Samples: {num_samples}")


Norm error: 3.448e-03 | Samples: 1000000


## 2. General Vector State Sketching

For arbitrary real vectors, QOS uses a Walsh-Hadamard randomization followed by LCU + QSVT arcsin inversion.

In [3]:
N = 128
key, subkey = random.split(key)
vector = random_unit_vector(subkey, N)

state, total_samples = q_state_sketch(vector, key, unit_num_samples=100_000, degree=20)
fidelity = jnp.abs(jnp.vdot(vector, state / jnp.linalg.norm(state)))**2
print(f"Fidelity: {float(fidelity):.4f} | Total samples: {total_samples}")


[sym_qsp] Iterative optimization to err 1.000e-12 or max_iter 100.
iter: 001 --- err: 1.484e-01
iter: 002 --- err: 3.077e-02
iter: 003 --- err: 4.367e-03
iter: 004 --- err: 1.753e-04
iter: 005 --- err: 3.352e-07
iter: 006 --- err: 1.235e-12
iter: 007 --- err: 6.674e-16
[sym_qsp] Stop criteria satisfied.


Fidelity: 0.9988 | Total samples: 1900000


## 3. Boolean Phase Oracle Sketching

Construct the phase oracle $\vert x \rangle \mapsto (-1)^{f(x)} \vert x \rangle$ from random truth-table queries.

In [4]:
N = 1000
key, subkey = random.split(key)
f = random.randint(subkey, (N,), minval=0, maxval=2)

diag, num_samples = q_oracle_sketch_boolean(f, unit_num_samples=10_000_000)
target = jnp.exp(1j * jnp.pi * f)
max_error = float(jnp.max(jnp.abs(diag - target)))
print(f"Max entrywise error: {max_error:.3e} | Samples: {num_samples}")


Max entrywise error: 4.929e-04 | Samples: 10000000


## 4. Sparse Matrix Element Oracle Sketching

Encode the oracle $\vert i \rangle\vert j \rangle \mapsto A_{ij} \vert i \rangle\vert j \rangle$.

In [5]:
N1, N2 = 100, 1000
nnz = 300
key, subkey = random.split(key)
A = random_sparse_matrix(subkey, (N1, N2), nnz)

oracle_diag, num_samples = q_oracle_sketch_matrix_element(A, unit_num_samples=1_000_000)
target = A.reshape(N1 * N2)
max_error = float(jnp.max(jnp.abs(oracle_diag - target)))
print(f"Max entrywise error: {max_error:.3e} | Samples: {num_samples}")


Max entrywise error: 6.987e-05 | Samples: 1000000


## 5. Sparse Matrix Row-Index Oracle Sketching

Encode the oracle that returns the column index of the k-th non-zero element in each row.

In [6]:
dim1, dim2 = 20, 50
nnz = 100
key, subkey = random.split(key)
A = random_sparse_matrix(subkey, (dim1, dim2), nnz)

index_oracle, num_samples = q_oracle_sketch_matrix_row_index(A, unit_num_samples=10_000_000)
print(f"Oracle shape: {index_oracle.shape} | Samples: {num_samples}")

# Verify first few rows
for i in range(min(3, dim1)):
    pred = jnp.argmax(jnp.abs(index_oracle[i, 0]))
    nz = jnp.nonzero(A[i])[0]
    print(f"Row {i}: predicted={pred}, actual={nz[0] if len(nz) else 'N/A'}")


Oracle shape: (20, 8, 50) | Samples: 10000000


Row 0: predicted=11, actual=11
Row 1: predicted=0, actual=0


Row 2: predicted=0, actual=0


## 6. Amplitude Amplification

Boost a low-norm unnormalized state to near-unit norm using QSVT.

In [7]:
dim = 100
key, subkey = random.split(key)
v = random_unit_vector(subkey, dim) * 0.3  # low norm

amplified = amplitude_amplification(v, degree=51, target_norm=0.99)
print(f"Original norm: {float(jnp.linalg.norm(v)):.3f}")
print(f"Amplified norm: {float(jnp.linalg.norm(amplified)):.4f}")
print(f"Direction error: {float(jnp.linalg.norm(v/jnp.linalg.norm(v) - amplified/jnp.linalg.norm(amplified))):.3e}")


[pyqsp.poly.PolySign] degree=51, delta=13.333333333333334
[PolyTaylorSeries] (Cheb) max 0.9896580999806533 is at 0.7885158719384558: normalizing
[PolyTaylorSeries] (Cheb) average error = 0.00043246066364104274 in the domain [-1, 1] using degree 51
[sym_qsp] Iterative optimization to err 1.000e-12 or max_iter 100.
iter: 001 --- err: 5.855e-01
iter: 002 --- err: 1.945e-01
iter: 003 --- err: 6.467e-02
iter: 004 --- err: 1.875e-02
iter: 005 --- err: 3.488e-03
iter: 006 --- err: 1.671e-04
iter: 007 --- err: 3.260e-07
iter: 008 --- err: 1.019e-12
iter: 009 --- err: 2.388e-15
[sym_qsp] Stop criteria satisfied.


Original norm: 0.300
Amplified norm: 0.9904
Direction error: 2.123e-15


## Next Steps

- Run the full synthetic benchmark: `python -m qos.experiments.benchmark`
- Explore real-dataset experiments in `examples/real_datasets/`
- Read the theory document at `docs/theory.md`
